# 第35课：MLOps 与 AI 工程化实践

## 学习目标
- 理解 MLOps 的核心思想：把 ML 模型从「实验」变成「产品」
- 掌握 ML 系统的完整生命周期：数据 → 训练 → 部署 → 监控 → 迭代
- 理解 LLM 时代 MLOps 的新挑战（prompt 版本管理、幻觉监控、成本控制）
- 了解关键工具链和最佳实践

## 为什么学这个？

前面 34 课，我们学了从线性回归到大模型微调、从 Prompt 工程到 AI 安全的完整技术栈。
但一个残酷的现实是：**训练出好模型只是 AI 工程的 20%，剩下 80% 是工程化**。

作为架构师，你一定深有体会——一个系统设计得再好，没有 CI/CD、没有监控、没有灰度发布，就不可能上线。
AI 系统也是一样，只是多了数据漂移、模型衰减、幻觉检测这些独特的挑战。

## 核心概念：MLOps 是什么？

### 直觉理解

把 DevOps 的思想搬到 ML 世界：

| DevOps | MLOps |
|--------|-------|
| 代码版本管理 | **代码 + 数据 + 模型** 版本管理 |
| CI/CD | **CT (Continuous Training)** + CI/CD |
| 服务监控 | **模型性能监控** + 服务监控 |
| 灰度发布 | **A/B 测试 + 影子模式** |
| 回滚 | **模型回滚 + 数据回滚** |

### MLOps 成熟度模型

```
Level 0: 手动流程（大多数团队在这里）
  → 手动训练、手动部署、无监控

Level 1: ML Pipeline 自动化
  → 自动训练、自动部署、基础监控

Level 2: CI/CD + CT
  → 代码变更触发训练、自动评估、自动部署
  → 数据变更也触发重训练

Level 3: 全自动化 + 治理
  → 特征商店、实验追踪、模型注册表、合规审计
```

### ML 系统的隐藏技术债

Google 著名论文《Hidden Technical Debt in ML Systems》指出：
ML 代码只占整个系统的很小一部分。周围有大量的基础设施：

```
┌─────────────────────────────────────────┐
│         配置 (Configuration)             │
│  ┌──────────────────────────────────┐   │
│  │       数据收集 (Data Collection)  │   │
│  │  ┌────────────────────────────┐  │   │
│  │  │    特征工程 (Features)      │  │   │
│  │  │  ┌──────────────────────┐  │  │   │
│  │  │  │   ML Code (很小!)    │  │  │   │
│  │  │  └──────────────────────┘  │  │   │
│  │  │    分析工具 (Analysis)      │  │   │
│  │  └────────────────────────────┘  │   │
│  │    资源管理 / Serving / 监控     │   │
│  └──────────────────────────────────┘   │
│         测试 / 进程管理                   │
└─────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 实践1: 模型版本管理与实验追踪（模拟）
# ============================================================
# 真实场景中用 MLflow / Weights & Biases / DVC
# 这里用简单数据结构演示核心概念

import hashlib
import json
import time
from datetime import datetime

class ExperimentTracker:
    """模拟 MLflow 风格的实验追踪"""
    
    def __init__(self):
        self.experiments = {}
    
    def log_experiment(self, name, params, metrics, model_path=None):
        """记录一次实验"""
        run_id = hashlib.md5(f"{name}{time.time()}".encode()).hexdigest()[:8]
        
        experiment = {
            'run_id': run_id,
            'name': name,
            'timestamp': datetime.now().isoformat(),
            'params': params,
            'metrics': metrics,
            'model_path': model_path,
            'status': 'completed'
        }
        
        self.experiments[run_id] = experiment
        return run_id
    
    def get_best_run(self, metric_name='accuracy', higher_is_better=True):
        """找到最佳实验"""
        best_run = None
        best_value = float('-inf') if higher_is_better else float('inf')
        
        for run in self.experiments.values():
            value = run['metrics'].get(metric_name, 0)
            if higher_is_better and value > best_value:
                best_value = value
                best_run = run
            elif not higher_is_better and value < best_value:
                best_value = value
                best_run = run
        
        return best_run

# 模拟 3 次实验
tracker = ExperimentTracker()

tracker.log_experiment(
    'sentiment-v1',
    params={'lr': 0.001, 'epochs': 10, 'batch_size': 32, 'model': 'bert-base'},
    metrics={'accuracy': 0.892, 'f1': 0.885, 'latency_ms': 45}
)

tracker.log_experiment(
    'sentiment-v1',
    params={'lr': 0.0005, 'epochs': 20, 'batch_size': 16, 'model': 'bert-base'},
    metrics={'accuracy': 0.918, 'f1': 0.912, 'latency_ms': 44}
)

tracker.log_experiment(
    'sentiment-v2',
    params={'lr': 0.001, 'epochs': 10, 'batch_size': 32, 'model': 'bert-large'},
    metrics={'accuracy': 0.935, 'f1': 0.928, 'latency_ms': 120}
)

best = tracker.get_best_run('accuracy')
print(f"🏆 最佳实验: {best['name']} (run: {best['run_id']})")
print(f"   准确率: {best['metrics']['accuracy']}")
print(f"   延迟: {best['metrics']['latency_ms']}ms")
print(f"\n⚠️ 注意: v2 准确率最高但延迟翻倍 → 需要权衡!")
print(f"   如果延迟预算 <100ms，应选 v1 的第二次实验 (acc=0.918)")

In [ ]:
# ============================================================
# 实践2: 模型部署策略模拟
# ============================================================
# 演示 4 种常见部署模式

import random

class ModelServer:
    """模拟模型服务"""
    def __init__(self, name, accuracy, latency_ms, cost_per_1k):
        self.name = name
        self.accuracy = accuracy
        self.latency_ms = latency_ms
        self.cost_per_1k = cost_per_1k
    
    def predict(self, text):
        # 模拟预测（随机结果，真实中会调用模型）
        time.sleep(self.latency_ms / 1000)  # 模拟延迟
        return {'label': random.choice(['positive', 'negative']), 
                'confidence': random.uniform(0.7, 0.99)}

class DeploymentManager:
    """模型部署管理器"""
    
    def __init__(self):
        self.versions = {}
        self.traffic_rules = {}
    
    def register(self, version, model):
        self.versions[version] = model
    
    def set_traffic(self, rules):
        """设置流量分配，如 {'v1': 0.8, 'v2': 0.2}"""
        self.traffic_rules = rules
    
    def route(self, request):
        """根据流量规则路由请求"""
        r = random.random()
        cumulative = 0
        for version, weight in self.traffic_rules.items():
            cumulative += weight
            if r <= cumulative:
                return self.versions[version]
        return list(self.versions.values())[-1]

# 创建两个模型版本
dm = DeploymentManager()
dm.register('v1', ModelServer('bert-base-finetuned', 0.918, 44, 0.002))
dm.register('v2', ModelServer('bert-large-finetuned', 0.935, 120, 0.008))

# 策略1: 蓝绿部署 (Blue-Green)
print("📦 策略1: 蓝绿部署")
print("   v1 和 v2 同时运行，流量瞬间切换")
dm.set_traffic({'v1': 0.0, 'v2': 1.0})  # 一键切到 v2
print("   当前流量: 100% → v2")

# 策略2: 金丝雀发布 (Canary)
print("\n🐦 策略2: 金丝雀发布")
dm.set_traffic({'v1': 0.95, 'v2': 0.05})  # 先给 v2 5% 流量
print("   第1小时: 5% → v2，监控错误率")
dm.set_traffic({'v1': 0.8, 'v2': 0.2})    # 逐步放量
print("   第2小时: 20% → v2")
dm.set_traffic({'v1': 0.5, 'v2': 0.5})
print("   第3小时: 50% → v2")

# 策略3: A/B 测试
print("\n🧪 策略3: A/B 测试")
print("   同时运行两个版本，收集数据对比")
print("   通常持续 1-2 周，确保统计显著性")

# 策略4: 影子模式 (Shadow)
print("\n👤 策略4: 影子模式")
print("   v2 接收真实流量但不返回结果")
print("   对比 v2 和 v1 的输出，验证无异常后上线")
print("\n💡 架构师建议: LLM 部署优先用影子模式 + 金丝雀")

In [ ]:
# ============================================================
# 实践3: 数据漂移检测（模拟）
# ============================================================
# 这是 MLOps 中最核心的监控能力之一

import numpy as np

class DriftDetector:
    """检测数据分布漂移"""
    
    def __init__(self, reference_stats):
        """保存训练时的参考统计"""
        self.ref_mean = reference_stats['mean']
        self.ref_std = reference_stats['std']
        self.ref_dist = reference_stats.get('distribution')
    
    def detect_mean_shift(self, new_data, threshold=2.0):
        """检测均值偏移（简单 z-score 方法）"""
        new_mean = np.mean(new_data)
        z_score = abs(new_mean - self.ref_mean) / (self.ref_std + 1e-8)
        
        return {
            'drifted': z_score > threshold,
            'z_score': round(z_score, 3),
            'ref_mean': round(self.ref_mean, 3),
            'new_mean': round(new_mean, 3),
            'shift': round(abs(new_mean - self.ref_mean), 3)
        }
    
    def detect_concept_drift(self, predictions, labels, threshold=0.05):
        """检测概念漂移：模型准确率突然下降"""
        accuracy = np.mean(predictions == labels)
        
        return {
            'drifted': accuracy < (1 - threshold),  # 假设参考准确率 ~1.0
            'current_accuracy': round(accuracy, 3),
            'threshold': threshold
        }

# 模拟场景：情感分析模型的输入分布变化
np.random.seed(42)

# 训练时：评论长度分布
ref_stats = {'mean': 50, 'std': 15}  # 平均50字，标准差15
detector = DriftDetector(ref_stats)

# 正常数据
normal_data = np.random.normal(50, 15, 1000)
result1 = detector.detect_mean_shift(normal_data)
print(f"✅ 正常数据: drifted={result1['drifted']}, z={result1['z_score']}")

# 漂移数据（用户开始写更长的评论）
drifted_data = np.random.normal(80, 20, 1000)  # 均值偏移到 80
result2 = detector.detect_mean_shift(drifted_data)
print(f"⚠️ 漂移数据: drifted={result2['drifted']}, z={result2['z_score']}")
print(f"   参考均值: {result2['ref_mean']} → 当前均值: {result2['new_mean']}")

# 概念漂移检测
labels = np.array([1]*100 + [0]*100)
good_preds = np.array([1]*97 + [0]*3 + [0]*95 + [1]*5)  # 96% 准确率
bad_preds = np.array([1]*70 + [0]*30 + [0]*60 + [1]*40)  # 65% 准确率

print(f"\n📊 概念漂移检测:")
r3 = detector.detect_concept_drift(good_preds, labels, threshold=0.10)
print(f"   正常模型: accuracy={r3['current_accuracy']}, drifted={r3['drifted']}")
r4 = detector.detect_concept_drift(bad_preds, labels, threshold=0.10)
print(f"   衰退模型: accuracy={r4['current_accuracy']}, drifted={r4['drifted']}")

print("\n💡 工程实践: 生产中通常用 KS检验、PSI、JSD 等统计方法")
print("   检测到漂移后，自动触发重训练 pipeline")

In [ ]:
# ============================================================
# 实践4: LLM 时代的 MLOps 新挑战
# ============================================================

class LLMMonitor:
    """LLM 特有的监控指标"""
    
    def __init__(self):
        self.metrics = {
            'total_requests': 0,
            'total_tokens_in': 0,
            'total_tokens_out': 0,
            'total_cost': 0.0,
            'hallucinations_detected': 0,
            'refusals': 0,
            'avg_latency_ms': 0,
            'p99_latency_ms': 0,
        }
        self.latencies = []
    
    def log_request(self, tokens_in, tokens_out, latency_ms, 
                    hallucination=False, refusal=False):
        self.metrics['total_requests'] += 1
        self.metrics['total_tokens_in'] += tokens_in
        self.metrics['total_tokens_out'] += tokens_out
        self.metrics['latencies'].append(latency_ms) if hasattr(self.metrics, 'latencies') else None
        self.latencies.append(latency_ms)
        
        # 简化的成本计算 (GPT-4 级别)
        cost = tokens_in * 0.00003 + tokens_out * 0.00006
        self.metrics['total_cost'] += cost
        
        if hallucination:
            self.metrics['hallucinations_detected'] += 1
        if refusal:
            self.metrics['refusals'] += 1
    
    def get_dashboard(self):
        total = self.metrics['total_requests']
        hall_rate = self.metrics['hallucinations_detected'] / max(total, 1) * 100
        refusal_rate = self.metrics['refusals'] / max(total, 1) * 100
        
        return {
            'requests': total,
            'total_cost': f"${self.metrics['total_cost']:.2f}",
            'tokens': f"{self.metrics['total_tokens_in'] + self.metrics['total_tokens_out']:,}",
            'hallucination_rate': f"{hall_rate:.1f}%",
            'refusal_rate': f"{refusal_rate:.1f}%",
            'avg_latency': f"{np.mean(self.latencies):.0f}ms" if self.latencies else 'N/A',
            'p99_latency': f"{np.percentile(self.latencies, 99):.0f}ms" if len(self.latencies) > 1 else 'N/A',
        }

# 模拟一天的生产流量
monitor = LLMMonitor()
np.random.seed(42)

for _ in range(1000):
    tokens_in = int(np.random.lognormal(4, 0.5))  # 输入 token 数
    tokens_out = int(np.random.lognormal(3.5, 0.7))
    latency = max(100, np.random.normal(800, 300))
    hallucination = np.random.random() < 0.03  # 3% 幻觉率
    refusal = np.random.random() < 0.02  # 2% 拒答率
    
    monitor.log_request(tokens_in, tokens_out, latency, hallucination, refusal)

print("📊 LLM 生产监控仪表盘")
print("=" * 40)
dashboard = monitor.get_dashboard()
for k, v in dashboard.items():
    print(f"  {k:20s}: {v}")

print("\n🚨 告警规则示例:")
print("  - 幻觉率 > 5% → 触发人工审查")
print("  - P99 延迟 > 3s → 自动降级到小模型")
print("  - 日成本超预算 20% → 通知 + 限流")
print("  - 拒答率 > 10% → 检查 safety filter 配置")

In [ ]:
# ============================================================
# 实践5: 端到端 ML Pipeline 设计（伪代码框架）
# ============================================================

class MLPipeline:
    """简化的 ML Pipeline 框架"""
    
    def __init__(self, name, config):
        self.name = name
        self.config = config
        self.stages = []
    
    def add_stage(self, name, fn, retry=3, timeout=300):
        self.stages.append({
            'name': name, 'fn': fn, 
            'retry': retry, 'timeout': timeout
        })
    
    def run(self):
        print(f"🚀 Pipeline [{self.name}] 开始执行")
        print(f"   配置: {json.dumps(self.config, indent=2)}")
        
        results = {}
        for stage in self.stages:
            stage_name = stage['name']
            try:
                print(f"\n  ▶ [{stage_name}] 执行中...")
                result = stage['fn'](self.config, results)
                results[stage_name] = result
                print(f"    ✅ [{stage_name}] 完成")
            except Exception as e:
                print(f"    ❌ [{stage_name}] 失败: {e}")
                raise
        
        print(f"\n✅ Pipeline [{self.name}] 执行完成!")
        return results

# 定义 Pipeline 各阶段
def validate_data(config, prev_results):
    print(f"    - 检查数据新鲜度")
    print(f"    - 数据量: {config.get('data_size', '100K rows')}")
    print(f"    - 特征数: {config.get('num_features', 128)}")
    return {'status': 'valid', 'rows': 100000}

def train_model(config, prev_results):
    print(f"    - 模型: {config.get('model_type', 'bert-base')}")
    print(f"    - Epochs: {config.get('epochs', 10)}")
    print(f"    - 学习率: {config.get('lr', 0.001)}")
    return {'status': 'trained', 'model_version': 'v35.2'}

def evaluate_model(config, prev_results):
    acc = 0.935
    print(f"    - 准确率: {acc}")
    print(f"    - F1 Score: 0.928")
    print(f"    - 通过阈值 ({config.get('acc_threshold', 0.90)}): {'✅' if acc >= 0.90 else '❌'}")
    return {'status': 'evaluated', 'accuracy': acc, 'passed': acc >= 0.90}

def deploy_model(config, prev_results):
    if not prev_results.get('evaluate_model', {}).get('passed', False):
        raise ValueError("模型未通过评估，拒绝部署!")
    print(f"    - 部署策略: {config.get('deploy_strategy', 'canary')}")
    print(f"    - 初始流量: 5%")
    return {'status': 'deployed', 'endpoint': '/api/v2/predict'}

# 构建并运行 Pipeline
pipeline = MLPipeline('sentiment-analysis', {
    'data_size': '100K rows',
    'model_type': 'bert-base',
    'epochs': 10,
    'lr': 0.001,
    'acc_threshold': 0.90,
    'deploy_strategy': 'canary'
})

pipeline.add_stage('validate_data', validate_data)
pipeline.add_stage('train_model', train_model)
pipeline.add_stage('evaluate_model', evaluate_model)
pipeline.add_stage('deploy_model', deploy_model)

result = pipeline.run()

print("\n" + "=" * 50)
print("🔗 关键工具链参考:")
print("  实验追踪: MLflow, W&B, Neptune")
print("  数据版本: DVC, LakeFS")
print("  Pipeline: Kubeflow, Airflow, Dagster")
print("  Serving: vLLM, TGI, Triton, BentoML")
print("  监控: Prometheus + Grafana, Evidently, Arize")

## 总结

### MLOps 关键概念速查表

| 概念 | 是什么 | 为什么重要 |
|------|--------|----------|
| 实验追踪 | 记录每次训练的参数/指标 | 可复现、可对比、可回溯 |
| 模型注册表 | 版本化管理训练好的模型 | 避免混乱，支持回滚 |
| CT (Continuous Training) | 数据/代码变更自动触发重训练 | 模型不陈旧 |
| 金丝雀发布 | 新模型先接收少量流量 | 降低上线风险 |
| 数据漂移检测 | 监控输入分布变化 | 提前发现模型衰退 |
| 幻觉监控 | 检测 LLM 输出的虚假内容 | LLM 时代的新刚需 |

### LLM 时代 vs 传统 ML 的 MLOps 差异

| 维度 | 传统 ML | LLM 时代 |
|------|---------|----------|
| 部署单位 | 训练好的模型权重 | Prompt + 模型 + RAG 管道 |
| 版本管理 | 模型版本 | Prompt 版本 + 模型版本 + 知识库版本 |
| 主要成本 | 训练成本 | **推理成本**（token 计费） |
| 监控重点 | 准确率、延迟 | 幻觉率、拒答率、成本、延迟 |
| 迭代速度 | 周级 | **天级甚至小时级**（改 prompt 即生效） |

### 关键论文 & 项目

- 📄 **Hidden Technical Debt in ML Systems** (Sculley et al., 2015) — ML 系统技术债的经典论文
- 📄 **MLOps: Continuous Delivery for ML** (Google, 2020) — MLOps 成熟度模型
- 🔧 **MLflow** — 开源实验追踪和模型管理平台
- 🔧 **vLLM** — 高性能 LLM 推理服务
- 🔧 **Evidently AI** — 数据漂移和模型监控

### 今天学完后你应该记住什么

1. **MLOps = DevOps + 数据管理 + 模型生命周期** — 不只是部署，是端到端
2. **ML 代码只是冰山一角** — 周围有大量基础设施需要建设
3. **LLM 时代 MLOps 的新变量**: prompt 版本管理、幻觉监控、token 成本控制
4. **部署策略**: 影子模式 → 金丝雀 → 全量，不要一步到位
5. **数据漂移是模型衰退的头号杀手** — 必须有检测机制

### 下一步

第36课将学习 **AI 应用架构与案例实战**，把前面学到的所有技术（Transformer、RAG、Agent、MLOps）串成一个完整的 AI 应用架构。